# Running a shell-wind (Type IIn) model with OpenStella-PARDISO

This note walks through the full chain: compiling the code, setting up
the initial conditions for a supernova shell colliding with a
circumstellar wind, what every input file contains, and how to launch
the run. The example is the `TypeIIn` model: a 7 Msun H/He shell at
10,000 km/s colliding with a steady 1/r^2 CSM wind, no inner ejecta,
no radioactive nickel.

## 1. Compiling

`build_parallel.sh` detects the host and sets up the toolchain:

- `viper*` / `raven*` (MPCDF): loads the site `intel` module, which
  provides `ifort` and `MKLROOT` together.
- Any other host: sources `setvars.sh` from the standard oneAPI
  locations if `MKLROOT` is not already set.
- `STELLA_SITE=viper|raven|generic` overrides detection.
- `ifx` is used automatically when `ifort` is not installed.

`NZON` must match the zone count of the model `.hyd` file; the binary
is dimensioned for exactly that many radial zones (`Mzon` in
`src/zone.inc`). Building with a smaller NZON than the model rejects
the input at startup.

In [ ]:
# Build for a 500-zone model (from the repo root)
bash build_parallel.sh 500

# Produces bin/xstella.exe (a symlink to run/strad/xstella6new.exe)

## 2. Initial conditions: shell + wind

The model is built outside STELLA and passed in through `.hyd`, `.abn`
and `.xni` files. For the TypeIIn model the generator is
`examples/typeiin/make_shell_wind_model.py` (standalone, numpy
only; identical physics to the SN2024pba pipeline's
`prepare_SN_B_model.py --no-ejecta`). The structure it
produces is:
  (X=0.7381, Y=0.2485). Density profile follows the Dessart 2010jl
  structure: an n_rho=8 power-law envelope over a flatter core, joined
  at v0=3000 km/s. Inner radius 1.3e12 cm, outer 4.3e14 cm.
- **Shell-wind buffer**: ~46 zones of geometric transition so the
  density slope stays below the grid limit (max |dlog rho/dlog r| = 8).
- **CSM wind**: steady 1/r^2 wind, Mdot=0.01 Msun/yr, v=100 km/s,
  T=2000 K, from 4.4e14 cm out to 1e16 cm (about 0.27 Msun of CSM).
- **No ejecta, no Ni56**: the inner boundary is the shell inner edge.
- **500 zones** total; STELLA starts the evolution at t_start=5 d and
  runs to TcurB=60 d.

Because the wind cannot hold its share of zones under the minimum
radial-spacing constraint, freed zones are redistributed to the shell;
the total always equals NZON exactly.

In [ ]:
# Standalone: examples/typeiin/make_shell_wind_model.py
# (numpy only, no pipeline dependency)
python examples/typeiin/make_shell_wind_model.py \
    --model-name TypeIIn \
    --output-dir ./TypeIIn_run \
    --shell-mass 7.0 --shell-velocity 10000 \
    --wind-mdot 0.01 --wind-velocity 100 \
    --wind-temperature 2000 --wind-r-outer 1e16 \
    --nzon 500 --t-start 5.0 --target-days 60.0

# Writes TypeIIn_run/{modmake,eve,vladsf,strad} with all
# inputs: .hyd .abn .xni .dat .eve eve.1 ronfict.1 strad.1


## 3. Input file formats

### `<model>.hyd` - structure

ASCII. One header line, then `nzon` zone rows, 8 columns each:

| Field | Content |
| ----- | ------- |
| header | `t_start[d]  nzon  mass_cut[Msun]  rcen[cm]  rho_cen[g/cc]` |
| per zone | `km  dMr[Msun]  r[cm]  rho[g/cc]  T[K]  u[cm/s]  Mr[Msun]  0` |

`mass_cut` is the mass interior to zone 1 (zero here). `dMr`/`Mr` are
zone mass and enclosed mass in solar units; `u` is the velocity STELLA
adopts when `EKO=0` in the `.dat` (EKO>0 would overwrite it with a
triangular profile - leave EKO=0 for an interaction model).

In [ ]:
head -3 TypeIIn.hyd
#    5.000E+00   500  0.00000E+00  6.48000E+11  9.81306E-10
#    1  3.9362E-06  1.2960000E+12  9.8131E-10  6.5000E+04  3.0000E+06  ...
#    2  3.8320E-07  1.3318010E+12  9.8131E-10  6.5000E+04  3.0842E+06  ...

### `<model>.abn` - composition

ASCII, one row per zone, 20 columns:

`km  0  0  0  H  He  C  N  O  Ne  Na  Mg  Al  Si  S  Ar  Ca  FePeak  Ni58  Ni56`

Mass fractions per zone. The TypeIIn model writes solar H, He, O, Si,
Ca, Fe and zeros elsewhere. Values must be consistent with what STELLA
expects for `Natom=15` (H, He, C, N, O, Ne, Na, Mg, Al, Si, S, Ar, Ca,
Fe, Ni); the extra columns carry Ni58/Ni56 bookkeeping.

### `<model>.xni` - nickel distribution

Fortran unformatted record: `2 + 2*nzon` REAL*4 floats wrapped in record
markers. A zero file (as here) disables radioactive decay; the pipeline
writes one automatically when `nickel56` is off.

### `<model>.dat` - run parameters

ASCII, labelled blocks. The settings that matter for this model:

| Parameter | Value | Meaning |
| --------- | ----- | ------- |
| `EPS` | 0.003 | stiff-integrator relative accuracy |
| `HMIN`/`HMAX` | 1e-14 / 3600 | min/max step (s) |
| `METH` | 3 | Gear+BGH stiff integrator |
| `TcurB` | 60.0 | stop time (days) |
| `NSTMAX` | 2000000 | max internal steps |
| `IOUT` | 10 | output cadence control |
| `EKO` | 0 | keep .hyd velocities (do NOT set >0) |
| `AMNI`/`XMNI` | 0/0 | no nickel |
| `AMHT`/`EBurst` | 0/0 | no central heating term |
| `FitTau` | 5.0 | Eddington fit tau |
| `SCAT` | T | electron scattering on |
| `NTO` + list | 61 values | fixed output epochs (days) for `.swd`/`.tt` |

The trailing `NTO` epoch list controls when structure snapshots and
photometry are written; spacing of ~1 day gives a smooth light curve.

### `strad.1` and `<model>.1` - run index and opacity

`strad.1` in the run directory maps run name to input files:

```text
Run             Results             Model            Nickel             Opacity
TypeIIn  TypeIIn.res  TypeIIn  TypeIIn  TypeIIn.1
```

`<model>.1` (and the `<model>.ab` sibling when the ronfict path is
used) is the precomputed opacity table; the pipeline generates it from
the vladsf tables via `generate_stella_opacity_tables.py`. `<model>.mod`
and `<model>.xni` in `strad/` are symlinks to the `eve/` outputs -
`.mod` is the STELLA-native binary structure file produced by
`eve2.exe` from `.hyd`/`.abn`.

## 4. Running the model

STELLA runs from the `strad/` directory and reads `strad.1` plus the
model files. Match threads to the allocation, then launch:

In [ ]:
# One-shot: eve2 -> xronfict -> xstella
bash examples/typeiin/run_model.sh ./TypeIIn_run 2

# Or manually, the equivalent steps:
export OMP_NUM_THREADS=2
export MKL_NUM_THREADS=2
export MKL_THREADING_LAYER=intel
export LD_LIBRARY_PATH="$MKLROOT/lib/intel64:$LD_LIBRARY_PATH"
export HOMEStella=/path/to/OpenStella_parallel

cd TypeIIn_run/strad
$HOMEStella/bin/xstella.exe < /dev/null


The `< /dev/null` matters: STELLA would otherwise block on stdin.

On a cluster this goes in a batch script with `--cpus-per-task` equal
to the thread count. For a 500-zone Type IIn plan on roughly 1-2 hours
with 2-3 threads.

## 5. Outputs

Written into the `strad/` run dir during the run and moved to
`$HOMEStella/res/` at the end:

| File | Content |
| ---- | ------- |
| `<model>.swd` | structure per output epoch: t, zone, mass, R, v, Tgas, Trad, rho, P, luminosity, kappa, n_bar, n_e, 15 mass fractions (30 cols) |
| `<model>.tt` | light curve: time, Tbb, Rbb, Teff, Mbol, 15 band magnitudes (UBVRI + ugriz + JHKs + co), Mbolavg, gdepos |
| `<model>.res` | epoch summary / model log |
| `<model>.lbol` | bolometric luminosity vs time |
| `<model>.ph` | multigroup spectra |
| `<model>.flx` | flux spectra for post-processing |
| `<model>.prf`/`.crv` | restart checkpoint (keep for RESTART mode) |

STELLA cannot overwrite existing `.swd`/`.tt`/`.res` files - delete or
back them up before a re-run, or the job exits immediately.